## 1. 필요한 라이브러리 불러오기

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

## 2. 하이퍼 파라미터 및 gpu 사용

MobileNet은 속도가 빠르므로 성능을 높이고자 보다 많은 epoch(=100)으로 설정하였다.

In [ ]:
# Hyper Parameters
IMAGE_SIZE = 224
LEARNING_RATE = 0.1
EPOCHS = 100
BATCH_SIZE = 128

# GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(torch.__version__)
print(device)

2.9.0+cu126
cuda


## 3. 데이터 준비

In [3]:
# 데이터 준비
cifar_mean = [0.4914, 0.4822, 0.4465]
cifar_std = [0.2023, 0.1994, 0.2010]

# Data Augmentation (Translation + Mirroring)
train_transform = transforms.Compose([
    # 1. Translation
    transforms.RandomCrop(32, padding=4),
    # 2. Mirroring
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_data = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_data = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:03<00:00, 48.6MB/s]


## 4. Depthwise Separable Convolution 클래스 정의

depthwise conv 부분에서 groups=in_channels으로 설정 시 입력 채널 1개 당 필터 1개가 독립적으로 매칭된다.

depthwise conv에서 이미지 크기에 대한 Downsampling이 진행되며
pointwise conv에서 채널 수를 변경한다.

또한 depthwise-BN-ReLU-Pointwise-BN-ReLU 구조로 구현하였다.

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super(DepthwiseSeparableConv, self).__init__()

        # 1. Depthwise Convolution (3x3, groups=in_channels)
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        # 2. Pointwise Convolution (1x1)
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

## 5. MobileNetV1 클래스 정의

 ImageNet 데이터셋을 사용한 MobileNet v1 논문에서는 처음에 이미지를 줄이고 시작하는 반면 여기선 CIFAR10 데이터셋을 사용했으므로 처음에 이미지를 줄이지 않고 구현하였다

In [ ]:
class MobileNetV1(nn.Module):
    def __init__(self, num_classes=10, alpha=1.0):
        super(MobileNetV1, self).__init__()

        # Define number of channels
        c32  = int(alpha * 32)
        c64  = int(alpha * 64)
        c128 = int(alpha * 128)
        c256 = int(alpha * 256)
        c512 = int(alpha * 512)
        c1024 = int(alpha * 1024)

        # Full Convolution
        # CIFAR-10 (32x32) -> Stride 1 -> 32x32
        # Full Convolution
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, c32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(c32), 
            nn.ReLU(inplace=True)
        )

        # Depthwise Separable Blocks
        self.layers = nn.Sequential(
            DepthwiseSeparableConv(c32, c64, stride=1),
            DepthwiseSeparableConv(c64, c128, stride=2),
            DepthwiseSeparableConv(c128, c128, stride=1),
            DepthwiseSeparableConv(c128, c256, stride=2),
            DepthwiseSeparableConv(c256, c256, stride=1),
            DepthwiseSeparableConv(c256, c512, stride=2), 
            DepthwiseSeparableConv(c512, c512, stride=1),
            DepthwiseSeparableConv(c512, c512, stride=1),
            DepthwiseSeparableConv(c512, c512, stride=1),
            DepthwiseSeparableConv(c512, c512, stride=1),
            DepthwiseSeparableConv(c512, c512, stride=1),
            DepthwiseSeparableConv(c512, c1024, stride=2),
            DepthwiseSeparableConv(c1024, c1024, stride=1)
        )

        # Average Pool, FC
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c1024, num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            # Conv Layer
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            # BN Layer  
            elif isinstance(m, nn.BatchNorm2d):
                
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
                
            # FC Layer
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        x = self.layers(x)
        x = self.avg_pool(x)
        x = torch.flatten(x, 1) # Flatten
        x = self.fc(x)
        return x

## 6. 훈련 준비 및 optimizer 선언

학습 스케쥴러 중 MultiStepLR를 사용한 이유는 단계적으로 학습률을 낮추어 효율적으로 학습시키기 위함이다.
실제 논문에서 사용되진 않았지만 학습 효율성을 높이기 위해 사용하였습니다.

In [ ]:
model = MobileNetV1(num_classes=10, alpha=1.0)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[30, 60], gamma=0.1)

## 7. 훈련

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/{원하는경로}'
os.makedirs(save_dir, exist_ok=True)

print(f"MobileNet_v1 학습 시작")
best_acc = 0
for epoch in range(EPOCHS):
    start_time = time.time()

    # training
    model.train()
    running_loss = 0.0

    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # scheduler 업데이트
    scheduler.step()

    # validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # 결과 계산
    epoch_loss = running_loss / len(train_dataloader)
    epoch_acc = 100 * correct / total
    epoch_time = time.time() - start_time

    # Best Model 저장 로직
    if epoch_acc > best_acc:
        best_acc = epoch_acc

        # Google Drive에 Best Model 저장
        save_path = os.path.join(save_dir, f"MobileNetV1_best.pth")
        torch.save(model.state_dict(), save_path)

    # 로그 출력
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {epoch_loss:.4f} | "
          f"Test Acc: {epoch_acc:.2f}% | "
          f"Time: {epoch_time:.1f}s")

print("MobileNet_v1 학습 종료")
print(f"최종 Best Accuracy: {best_acc:.2f}%")

Mounted at /content/drive
MobileNet_v1 학습 시작
Epoch [1/100] Loss: 1.7959 | Test Acc: 45.07% | Time: 29.7s
Epoch [2/100] Loss: 1.3744 | Test Acc: 57.66% | Time: 29.1s
Epoch [3/100] Loss: 1.1267 | Test Acc: 61.87% | Time: 27.5s
Epoch [4/100] Loss: 0.9326 | Test Acc: 67.88% | Time: 29.0s
Epoch [5/100] Loss: 0.8365 | Test Acc: 68.99% | Time: 25.0s
Epoch [6/100] Loss: 0.7721 | Test Acc: 62.68% | Time: 25.5s
Epoch [7/100] Loss: 0.7265 | Test Acc: 71.15% | Time: 26.0s
Epoch [8/100] Loss: 0.7093 | Test Acc: 73.68% | Time: 26.1s
Epoch [9/100] Loss: 0.6824 | Test Acc: 71.21% | Time: 25.4s
Epoch [10/100] Loss: 0.6663 | Test Acc: 73.32% | Time: 25.0s
Epoch [11/100] Loss: 0.6643 | Test Acc: 73.80% | Time: 24.6s
Epoch [12/100] Loss: 0.6559 | Test Acc: 72.08% | Time: 25.5s
Epoch [13/100] Loss: 0.6537 | Test Acc: 75.80% | Time: 25.5s
Epoch [14/100] Loss: 0.6426 | Test Acc: 72.98% | Time: 27.9s
Epoch [15/100] Loss: 0.6419 | Test Acc: 69.02% | Time: 25.8s
Epoch [16/100] Loss: 0.6418 | Test Acc: 70.44% | 